# Project 1 — Classifying user requests

We compare three text representations and three classifiers on filtered WildGuardMix prompts. The steps follow the [course tutorial](https://github.com/nlp-illc/nlp1-student/blob/6d93c951fa2bd0a92b6e90ca954865f1c1165eef/part1/tutorial.ipynb).

Run [setup.ipynb](setup.ipynb) first, then run this notebook from top to bottom. Repeated encoding and training code is in [functions.py](../utils/p1/functions.py).

## 1. Setup

Import the course code and project helpers. We use a fixed seed and CPU training.

In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "part1/smlp.py").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

# Keep the normal Hugging Face login token location. Redirecting XDG_CACHE_HOME
# below would otherwise make huggingface_hub look for the token inside root/.cache.
default_hf_home = Path(os.environ.get("HF_HOME") or
                       Path(os.environ.get("XDG_CACHE_HOME") or Path.home() / ".cache") / "huggingface")
os.environ.setdefault("HF_TOKEN_PATH", str(default_hf_home / "token"))

# Keep generated caches in this local, Git-ignored directory.
os.environ.setdefault("MPLCONFIGDIR", str(root / ".cache/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(root / ".cache"))
os.environ.setdefault("HF_HUB_CACHE", str(root / ".cache/huggingface/hub"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [ ]:
import hashlib
import json
import random
import time
from importlib.metadata import version
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

import part1
from utils.p1.functions import (
    TokenIndicators, BagOfWords, TFIDF, encode_texts,
    TwoHiddenMLP, train_mlp, predict_mlp, classification_scores,
)

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

### Settings

Use 4,000 prompts per class, five sample seeds, a 3,000-token BPE vocabulary and 10 epochs. Set `use_toy_data = True` for an offline pipeline check with artificial topic labels. Toy scores do not measure harmfulness classification.

In [ ]:
use_toy_data = False
seed = 42
num_runs = 5  # independent balanced samples; each gets its own 5-fold CV
num_folds = 5
examples_per_class = 4000
vocab_size = 3000
min_frequency = 2
num_epochs = 10
hidden_dim = 64
second_hidden_dim = 32
dropout = 0.2
batch_size = 64
learning_rate = 0.001
weight_decay = 0.0001
rope = 0.01

dataset_revision = "d29c47f41c8b51348b5c8e8c81c039b3132b66d1"
if use_toy_data:
    examples_per_class, vocab_size, num_epochs, num_runs = 20, 128, 2, 2

# Run 0 uses the original seed, so it reproduces the single-sample results.
# Steps of 100 keep the per-fold model seeds (sample_seed + fold) distinct across runs.
sample_seeds = [seed + 100 * run for run in range(num_runs)]

run_label = "SMOKE_NOT_SCIENTIFIC" if use_toy_data else "PROJECT1_CV"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
results_folder = root / "part1/results" / ("toy" if use_toy_data else "project1") / run_id
results_folder.mkdir(parents=True, exist_ok=False)
print(run_label, "| sample seeds:", sample_seeds)

## 2. Load the data

Predict `prompt_harm_label` from the prompt text only: 0 = unharmful, 1 = harmful. Load the pinned training split using your Hugging Face login. The official test stays held out until the final evaluation in section 14.

In [ ]:
if use_toy_data:
    records = []
    for i in range(examples_per_class):
        records.append({"prompt": f"Garden notebook {i}: water the flowers and prepare fresh soil.",
                        "prompt_harm_label": "unharmful", "subcategory": "synthetic"})
        records.append({"prompt": f"Space notebook {i}: observe the planets through a telescope tonight.",
                        "prompt_harm_label": "harmful", "subcategory": "synthetic"})
    data = pd.DataFrame(records)
else:
    dataset = load_dataset(
        "allenai/wildguardmix", "wildguardtrain", split="train",
        revision=dataset_revision, token=True,
        cache_dir=str(root / ".cache/huggingface/datasets"),
    )
    assert dataset.cache_files and all(
        dataset_revision in Path(item["filename"]).parts for item in dataset.cache_files
    ), "The cached dataset does not match our pinned revision."
    data = dataset.select_columns(["prompt", "prompt_harm_label", "subcategory"]).to_pandas()

print("Rows loaded:", len(data))

### Filter the agreed category

Remove `toxic_language_hate_speech` using an exact match after stripping whitespace and ignoring case. Then check labels and the expected dataset counts.

In [ ]:
audit = {"status": run_label, "original_rows": len(data)}
excluded = data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech")
audit["excluded_category_rows"] = int(excluded.sum())
data = data.loc[~excluded].copy()
assert not data["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech").any()

labels = data["prompt_harm_label"].str.strip().str.casefold()
missing_label = labels.isna() | labels.eq("")
audit["missing_label_rows"] = int(missing_label.sum())
assert labels.loc[~missing_label].isin(["unharmful", "harmful"]).all(), "Unexpected label."
data = data.loc[~missing_label].copy()
data["label"] = labels.loc[~missing_label].map({"unharmful": 0, "harmful": 1}).astype(int)
audit["eligible_rows"] = len(data)
audit["eligible_unharmful"] = int(data["label"].eq(0).sum())
audit["eligible_harmful"] = int(data["label"].eq(1).sum())

if not use_toy_data:
    expected = {"original_rows": 86759, "excluded_category_rows": 3020,
                "missing_label_rows": 0, "eligible_rows": 83739,
                "eligible_unharmful": 40543, "eligible_harmful": 43196}
    assert all(audit[key] == value for key, value in expected.items()), "Dataset audit mismatch."
display(pd.Series(audit, name="Data audit"))

### Clean and deduplicate

Normalize Unicode and trim edge whitespace. Remove empty prompts, all copies of prompts with conflicting labels, and consistent duplicates. We count each removal separately. Related scenarios may still cross folds.

In [ ]:
data["prompt"] = data["prompt"].fillna("").str.normalize("NFC").str.strip()
empty_prompt = data["prompt"].eq("")
audit["empty_prompt_rows"] = int(empty_prompt.sum())
data = data.loc[~empty_prompt].copy()

conflicting = data.groupby("prompt")["label"].transform("nunique").gt(1)
audit["contradictory_label_rows"] = int(conflicting.sum())
audit["contradictory_label_groups"] = int(data.loc[conflicting, "prompt"].nunique())
data = data.loc[~conflicting].copy()
audit["duplicate_rows"] = int(data.duplicated("prompt").sum())
data = data.drop_duplicates("prompt").copy()
audit["retained_rows"] = len(data)

removals = ["excluded_category_rows", "missing_label_rows", "empty_prompt_rows",
            "contradictory_label_rows", "duplicate_rows", "retained_rows"]
assert sum(audit[key] for key in removals) == audit["original_rows"]
assert data["prompt"].is_unique

data["id"] = data["prompt"].map(lambda text: "wg-" + hashlib.sha256(text.encode("utf-8")).hexdigest())
data = data[["id", "prompt", "label"]].sort_values("id").reset_index(drop=True)
display(pd.Series(audit, name="Completed audit"))

### Sample both classes

Take an equal number from each class using a fixed seed. The resulting class balance differs from the source population. Hash IDs let us match predictions without exporting prompt text.

In [ ]:
def sample_balanced(data, sample_seed):
    """Draw examples_per_class prompts per class, reproducibly for a given seed."""
    rng = random.Random(sample_seed)
    chosen_indices = []
    for label in [0, 1]:
        candidates = data.index[data["label"].eq(label)].tolist()
        assert len(candidates) >= examples_per_class
        chosen_indices.extend(rng.sample(candidates, examples_per_class))
    rng.shuffle(chosen_indices)
    return data.loc[chosen_indices].reset_index(drop=True)

# The walkthrough in sections 3-8 uses the first sample (the original seed).
sample = sample_balanced(data, sample_seeds[0])
texts = sample["prompt"].tolist()
y = sample["label"].to_numpy()
ids = sample["id"].to_numpy()
audit["sample_rows"] = len(sample)
audit["sample_harmful_prevalence"] = float(y.mean())
print("Sample class counts:", sample["label"].value_counts().sort_index().to_dict())
print("Retained population harmful prevalence:", data["label"].mean())
display(sample["prompt"].str.len().describe(percentiles=[0.5, 0.95]).rename("Prompt length in characters"))

# How much do the samples overlap? Low overlap means the runs are close to independent draws.
sample_id_sets = [set(sample_balanced(data, s)["id"]) for s in sample_seeds]
overlap = [len(sample_id_sets[0] & other) / len(sample_id_sets[0]) for other in sample_id_sets[1:]]
print("Overlap of run 0 with the other runs:", [round(o, 3) for o in overlap])

## 3. Create five folds

Use the same stratified folds for every system. BPE and IDF are fitted inside each training fold. We start with fold 0 to walk through the pipeline.

In [ ]:
def make_folds(y, sample_seed):
    splitter = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=sample_seed)
    folds = list(splitter.split(np.zeros(len(y)), y))
    for train_indices, validation_indices in folds:
        assert set(train_indices).isdisjoint(validation_indices)
    return folds

folds = make_folds(y, sample_seeds[0])
for fold, (train_indices, validation_indices) in enumerate(folds):
    print(f"Fold {fold + 1}: {len(train_indices)} train, {len(validation_indices)} validation")

# Walk through the pipeline on fold 0 first.
train_indices, validation_indices = folds[0]
train_texts = [texts[i] for i in train_indices]
validation_texts = [texts[i] for i in validation_indices]
y_train, y_validation = y[train_indices], y[validation_indices]

## 4. Train the BPE tokenizer

Reuse the course tokenizer on training text only. BPE learns frequent character merges; its tokens can be words or smaller pieces. Inspect a safe example below.

In [ ]:
tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
safe_sentence = "Please explain how plants grow in a garden."
print(tokenizer.encode(safe_sentence).tokens)
print("Learned vocabulary size:", tokenizer.get_vocab_size())

## 5. Build three representations

- **Indicators:** whether each token appears.
- **Counts:** how often each token appears.
- **TF-IDF:** counts weighted by training-document rarity, then L2-normalized.

All three share the same BPE tokenizer and discard token order. Our smoothed IDF is `log((1 + n_train) / (1 + df_train)) + 1`.

In [ ]:
indicator_encoder = TokenIndicators(tokenizer)
bow_encoder = BagOfWords(tokenizer)
tfidf_encoder = TFIDF(tokenizer).fit(train_texts)
encoders = {"indicators": indicator_encoder, "bow": bow_encoder, "tfidf": tfidf_encoder}

safe_repetition = "garden garden garden plants"
token_ids = np.flatnonzero(bow_encoder(safe_repetition))
display(pd.DataFrame({
    "token": [tokenizer.id_to_token(int(i)) for i in token_ids],
    "indicator": indicator_encoder(safe_repetition)[token_ids],
    "count": bow_encoder(safe_repetition)[token_ids],
    "tfidf": tfidf_encoder(safe_repetition)[token_ids],
}))
assert bow_encoder(safe_repetition).max() > 1
assert set(indicator_encoder(safe_repetition)).issubset({0, 1})

### Encode the first fold

Use TF-IDF for the next three model demonstrations. Transforming validation text must leave the vocabulary and IDF unchanged.

In [ ]:
vocabulary_before = tokenizer.get_vocab().copy()
idf_before = tfidf_encoder.transformer.idf_.copy()
X_train = encode_texts(train_texts, tfidf_encoder)
X_validation = encode_texts(validation_texts, tfidf_encoder)
assert tokenizer.get_vocab() == vocabulary_before
np.testing.assert_array_equal(tfidf_encoder.transformer.idf_, idf_before)
print("Feature matrix shapes:", X_train.shape, X_validation.shape)

## 6. Logistic regression

Fit a regularized linear classifier to the feature matrix. Probability columns are `[unharmful, harmful]`; predict the more probable class.

In [ ]:
logistic_model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=seed)
logistic_model.fit(X_train, y_train)
logistic_probabilities = logistic_model.predict_proba(X_validation)
logistic_predictions = logistic_probabilities.argmax(axis=1)
first_fold_scores = {"logistic": classification_scores(y_validation, logistic_predictions)}
display(pd.Series(first_fold_scores["logistic"]))

## 7. One hidden layer

Reuse the course sparse MLP with 64 ReLU units and dropout. Train with Adam and negative log-likelihood for a fixed number of epochs. Validation does not select a checkpoint.

In [ ]:
torch.manual_seed(seed)
mlp_model = part1.smlp.SparseMLP(X_train.shape[1], hidden_dim, num_classes=2, dropout=dropout)
training_history = train_mlp(
    mlp_model, X_train, y_train, epochs=num_epochs, seed=seed,
    batch_size=batch_size, learning_rate=learning_rate, weight_decay=weight_decay,
)
mlp_probabilities = predict_mlp(mlp_model, X_validation)
first_fold_scores["mlp1"] = classification_scores(y_validation, mlp_probabilities.argmax(axis=1))
display(pd.DataFrame(training_history).rename_axis("epoch"))

## 8. Two hidden layers

Extend the course MLP to 64 → 32 ReLU units. Keep the optimizer, dropout and epoch budget fixed. This changes both depth and parameter count.

In [ ]:
torch.manual_seed(seed)
modified_mlp = TwoHiddenMLP(X_train.shape[1], hidden_dim, second_hidden_dim, dropout)
modified_history = train_mlp(
    modified_mlp, X_train, y_train, epochs=num_epochs, seed=seed,
    batch_size=batch_size, learning_rate=learning_rate, weight_decay=weight_decay,
)
modified_probabilities = predict_mlp(modified_mlp, X_validation)
first_fold_scores["mlp2"] = classification_scores(y_validation, modified_probabilities.argmax(axis=1))
display(pd.DataFrame(first_fold_scores).T)
print("These are first-fold demonstrations. The complete comparison comes next.")

## 9. Run the full comparison

Train all nine representation/model combinations on `num_runs` independent balanced samples, each with its own five folds: 9 × 5 × 5 = 225 fits. Each fold gets a fresh tokenizer and IDF. Run 0 uses the original seed and reproduces the single-sample results. Save held-out predictions and report macro-F1, accuracy, harmful recall and benign false-positive rate. Fit time excludes feature construction.

In [ ]:
score_rows = []
prediction_rows = []
fold_rows = []
model_names = ["logistic", "mlp1", "mlp2"]
representation_names = ["indicators", "bow", "tfidf"]
started_all = time.perf_counter()

for run, sample_seed in enumerate(sample_seeds):
    run_sample = sample_balanced(data, sample_seed)
    run_texts = run_sample["prompt"].tolist()
    run_y = run_sample["label"].to_numpy()
    run_ids = run_sample["id"].to_numpy()
    run_folds = make_folds(run_y, sample_seed)

    for fold, (train_indices, validation_indices) in enumerate(run_folds):
        fold_seed = sample_seed + fold
        train_texts = [run_texts[i] for i in train_indices]
        validation_texts = [run_texts[i] for i in validation_indices]
        y_train, y_validation = run_y[train_indices], run_y[validation_indices]
        fold_rows.extend({"run": run, "id": run_ids[i], "gold": int(run_y[i]), "fold": fold}
                         for i in validation_indices)

        tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
        encoders = {
            "indicators": TokenIndicators(tokenizer),
            "bow": BagOfWords(tokenizer),
            "tfidf": TFIDF(tokenizer).fit(train_texts),
        }
        for representation, encoder in encoders.items():
            X_train = encode_texts(train_texts, encoder)
            X_validation = encode_texts(validation_texts, encoder)

            for model_name in model_names:
                started = time.perf_counter()
                if model_name == "logistic":
                    model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=fold_seed)
                    model.fit(X_train, y_train)
                    fit_seconds = time.perf_counter() - started
                    probabilities = model.predict_proba(X_validation)
                else:
                    torch.manual_seed(fold_seed)
                    if model_name == "mlp1":
                        model = part1.smlp.SparseMLP(X_train.shape[1], hidden_dim, 2, dropout)
                    else:
                        model = TwoHiddenMLP(X_train.shape[1], hidden_dim, second_hidden_dim, dropout)
                    history = train_mlp(model, X_train, y_train, epochs=num_epochs, seed=fold_seed,
                                        batch_size=batch_size, learning_rate=learning_rate,
                                        weight_decay=weight_decay)
                    fit_seconds = time.perf_counter() - started
                    probabilities = predict_mlp(model, X_validation)

                assert np.isfinite(probabilities).all()
                np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)
                predictions = probabilities.argmax(axis=1)
                metrics = classification_scores(y_validation, predictions)
                score_rows.append({"status": run_label, "run": run, "sample_seed": sample_seed,
                                   "model": model_name, "representation": representation, "fold": fold,
                                   **metrics, "fit_seconds": fit_seconds, "seed": fold_seed})
                for j, index in enumerate(validation_indices):
                    prediction_rows.append({"run": run, "id": run_ids[index], "model": model_name,
                        "representation": representation, "fold": fold, "gold": int(run_y[index]),
                        "prediction": int(predictions[j]), "confidence": float(probabilities[j].max()),
                        "probability_harmful": float(probabilities[j, 1]), "status": run_label})
        print(f"Run {run + 1}/{num_runs}, fold {fold + 1}/{num_folds} done "
              f"({time.perf_counter() - started_all:.0f}s elapsed)")

scores = pd.DataFrame(score_rows)
oof = pd.DataFrame(prediction_rows)
fold_table = pd.DataFrame(fold_rows)
num_systems = len(model_names) * len(representation_names)
assert len(scores) == num_runs * num_folds * num_systems
assert len(oof) == num_runs * 2 * examples_per_class * num_systems
assert not oof.duplicated(["run", "id", "model", "representation"]).any()
assert oof.groupby(["run", "id"])["fold"].nunique().eq(1).all()
print(run_label, f"— completed all {len(scores)} fits")

## 10. Compare scores

Average all held-out scores per system (runs × folds). The standard deviation now covers both fold and sample variation; overlapping training folds still mean it is not an independent-sample confidence interval.

We also show the mean per run and how each system ranks within each run. If the top system changes between samples, the observed ranking should not be read as a real difference.

In [ ]:
summary = scores.groupby(["model", "representation"]).agg(
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
    accuracy=("accuracy", "mean"), harmful_recall=("harmful_recall", "mean"),
    benign_fpr=("benign_fpr", "mean"), fit_seconds=("fit_seconds", "mean"),
).reset_index()
display(summary.round(4))

# Mean macro-F1 per run: how much does the sample alone move each system?
scores["system"] = scores["model"] + "/" + scores["representation"]
per_run = scores.pivot_table(index="system", columns="run", values="macro_f1", aggfunc="mean")
per_run["spread_between_runs"] = per_run.max(axis=1) - per_run.min(axis=1)
display(per_run.round(4))

# Rank of each system within each run (1 = best mean macro-F1 in that run).
ranks = per_run.drop(columns="spread_between_runs").rank(ascending=False).astype(int)
ranks["mean_rank"] = ranks.mean(axis=1)
display(ranks.sort_values("mean_rank"))
print("Leader per run:", per_run.drop(columns="spread_between_runs").idxmax().to_dict())

plot_data = scores.assign(model=scores["system"])
part1.analysis.plot_box_and_points(plot_data, metric="macro_f1")

## 11. Bayesian comparison

Compare logistic indicators vs logistic TF-IDF, then logistic vs one-layer MLP on TF-IDF. These were the two contrasts chosen for the original study.

The course model uses **B − A**: positive values favour B. Use ROPE **±0.01 macro-F1**, treating a one-point difference as small for this project. With `runs = num_runs`, the correlated t-test uses all runs × folds differences (paired by run and fold) and corrects for the overlap between training folds. Constant differences use a point-mass convention instead of a zero-scale t distribution.

In [ ]:
comparisons = [
    (("logistic", "indicators"), ("logistic", "tfidf")),
    (("logistic", "tfidf"), ("mlp1", "tfidf")),
]
comparison_rows = []

for system_a, system_b in comparisons:
    a = scores[(scores.model == system_a[0]) & (scores.representation == system_a[1])].sort_values(["run", "fold"])
    b = scores[(scores.model == system_b[0]) & (scores.representation == system_b[1])].sort_values(["run", "fold"])
    assert a[["run", "fold"]].values.tolist() == b[["run", "fold"]].values.tolist()
    differences = b["macro_f1"].to_numpy() - a["macro_f1"].to_numpy()
    names = ["/".join(system_a), "/".join(system_b)]

    if np.allclose(differences, differences[0]):
        mean_difference = float(differences.mean())
        equivalent = abs(mean_difference) <= rope or np.isclose(abs(mean_difference), rope)
        p_a = float(mean_difference < -rope and not equivalent)
        p_tie = float(equivalent)
        p_b = float(mean_difference > rope and not equivalent)
        fig, ax = plt.subplots()
        ax.axvspan(-rope, rope, alpha=0.2, color="teal", label="ROPE")
        ax.axvline(mean_difference, label="Point mass: identical fold differences")
        ax.set_xlabel("Mean macro-F1 difference B - A")
        ax.legend()
    else:
        posterior = part1.analysis.BenavoliCorrelatedT(
            a["macro_f1"].to_numpy(), b["macro_f1"].to_numpy(), runs=num_runs, names=names)
        p_a, p_tie, p_b = posterior.probs(rope)
        mean_difference = float(posterior.mean_diff)
        fig, ax = posterior.plot(rope)
        assert np.isclose(mean_difference, differences.mean())

    ax.set_title(f"{run_label}\nA: {names[0]} vs B: {names[1]}")
    fig.tight_layout()
    fig.savefig(results_folder / f"rope_{len(comparison_rows) + 1}.png", dpi=150, bbox_inches="tight")
    plt.show()
    assert np.isclose(p_a + p_tie + p_b, 1)
    comparison_rows.append({"A": names[0], "B": names[1], "mean_B_minus_A": mean_difference,
                            "rope": rope, "P_A_wins": p_a, "P_equivalent": p_tie, "P_B_wins": p_b})

bayesian_results = pd.DataFrame(comparison_rows)
display(bayesian_results.round(4))

## 12. Inspect errors

Inspect the system with the highest observed mean F1, pooled over all runs (a prompt drawn in several runs counts once per run). Count false positives, false negatives and errors with predicted-class probability ≥ 0.8. This ranking is descriptive, and probabilities are uncalibrated. Display hashed IDs rather than raw prompts.

In [ ]:
best = summary.loc[summary["macro_f1_mean"].idxmax()]
best_predictions = oof[(oof.model == best["model"]) & (oof.representation == best["representation"])].copy()
print("Descriptive CV leader:", best["model"], best["representation"])
counts = classification_scores(best_predictions["gold"], best_predictions["prediction"])
display(pd.DataFrame([[counts["tn"], counts["fp"]], [counts["fn"], counts["tp"]]],
                     index=["Gold unharmful", "Gold harmful"],
                     columns=["Predicted unharmful", "Predicted harmful"]))

errors = best_predictions[best_predictions.gold != best_predictions.prediction].copy()
errors["error_type"] = np.where(errors.gold == 0, "false_positive", "false_negative")
errors["confidently_wrong"] = errors["confidence"] >= 0.8
display(errors.groupby("error_type").agg(errors=("id", "count"), confidently_wrong=("confidently_wrong", "sum")))
display(errors.sort_values("confidence", ascending=False).head(10))

## 13. Save this run

Save metrics, hashed-ID predictions, split membership, settings and audit counts in this run’s local results folder. A new folder is created each run so previous results remain available.

In [ ]:
scores.to_csv(results_folder / "p1_cv_scores.csv", index=False)
summary.to_csv(results_folder / "p1_summary.csv", index=False)
per_run.to_csv(results_folder / "p1_per_run.csv")
ranks.to_csv(results_folder / "p1_ranks.csv")
oof.to_csv(results_folder / "p1_oof_predictions.csv", index=False)
errors.to_csv(results_folder / "p1_errors.csv", index=False)
bayesian_results.to_csv(results_folder / "p1_rope.csv", index=False)
fold_table.to_csv(results_folder / "folds.csv", index=False)
(results_folder / "data_audit.json").write_text(json.dumps(audit, indent=2))

settings = {"run_id": run_id, "status": run_label, "seed": seed, "folds": num_folds,
            "runs": num_runs, "sample_seeds": sample_seeds,
            "examples_per_class": examples_per_class, "vocab_size": vocab_size,
            "min_frequency": min_frequency, "epochs": num_epochs,
            "hidden_dim": hidden_dim, "second_hidden_dim": second_hidden_dim,
            "dropout": dropout, "batch_size": batch_size, "learning_rate": learning_rate,
            "weight_decay": weight_decay, "logistic_C": 1.0, "logistic_solver": "lbfgs",
            "rope": rope, "dataset_revision": None if use_toy_data else dataset_revision,
            "official_test_accessed": False,
            "packages": {name: version(name) for name in
                         ["numpy", "pandas", "scipy", "scikit-learn", "torch", "tokenizers", "datasets", "baycomp"]}}
(results_folder / "settings.json").write_text(json.dumps(settings, indent=2))
print("Saved:", results_folder.relative_to(root))

## 14. Final evaluation on the official test set

All settings above were fixed before looking at the test set, and nothing is tuned on it. For each sample seed, train each of the nine systems once on the full balanced sample (no folds; tokenizer and IDF fitted on that sample only) and predict WildGuardTest once. The spread over seeds reflects only the training sample and initialization; the test set itself is fixed.

The test set gets the same category filter and cleaning as the training data, and any prompt that also occurs in the training pool is removed. Its harmful share (~43%) differs from our balanced training samples, so watch benign FPR. Macro-F1 is also split into adversarial and vanilla prompts.

The two pre-chosen contrasts are compared with a paired bootstrap over test prompts (B − A, averaged over training runs), using the same ROPE as section 11.

This section only needs sections 1–2 and the sampling cell, not the CV results. **Run it once, and do not change settings based on its scores.**

In [ ]:
if use_toy_data:
    test_records = []
    for i in range(50):
        test_records.append({"prompt": f"Garden test {i}: plant seeds and water the soil.",
                             "prompt_harm_label": "unharmful", "subcategory": "synthetic", "adversarial": i % 2 == 0})
        test_records.append({"prompt": f"Space test {i}: watch the planets and the stars tonight.",
                             "prompt_harm_label": "harmful", "subcategory": "synthetic", "adversarial": i % 2 == 0})
    test_raw = pd.DataFrame(test_records)
else:
    test_dataset = load_dataset(
        "allenai/wildguardmix", "wildguardtest", split="test",
        revision=dataset_revision, token=True,
        cache_dir=str(root / ".cache/huggingface/datasets"),
    )
    assert test_dataset.cache_files and all(
        dataset_revision in Path(item["filename"]).parts for item in test_dataset.cache_files
    ), "The cached test set does not match our pinned revision."
    test_raw = test_dataset.select_columns(
        ["prompt", "prompt_harm_label", "subcategory", "adversarial"]).to_pandas()

# Exactly the same filtering and cleaning as the training data.
test_audit = {"status": run_label, "original_rows": len(test_raw)}
excluded = test_raw["subcategory"].fillna("").str.strip().str.casefold().eq("toxic_language_hate_speech")
test_audit["excluded_category_rows"] = int(excluded.sum())
test_data = test_raw.loc[~excluded].copy()

labels = test_data["prompt_harm_label"].fillna("").str.strip().str.casefold()
missing_label = labels.eq("")
test_audit["missing_label_rows"] = int(missing_label.sum())
assert labels.loc[~missing_label].isin(["unharmful", "harmful"]).all(), "Unexpected test label."
test_data = test_data.loc[~missing_label].copy()
test_data["label"] = labels.loc[~missing_label].map({"unharmful": 0, "harmful": 1}).astype(int)

test_data["prompt"] = test_data["prompt"].fillna("").str.normalize("NFC").str.strip()
empty_prompt = test_data["prompt"].eq("")
test_audit["empty_prompt_rows"] = int(empty_prompt.sum())
test_data = test_data.loc[~empty_prompt].copy()
conflicting = test_data.groupby("prompt")["label"].transform("nunique").gt(1)
test_audit["contradictory_label_rows"] = int(conflicting.sum())
test_data = test_data.loc[~conflicting].copy()
test_audit["duplicate_rows"] = int(test_data.duplicated("prompt").sum())
test_data = test_data.drop_duplicates("prompt").copy()

# Remove any test prompt that also occurs in the cleaned training pool.
test_data["id"] = test_data["prompt"].map(lambda text: "wg-" + hashlib.sha256(text.encode("utf-8")).hexdigest())
in_train = test_data["id"].isin(data["id"])
test_audit["train_overlap_rows"] = int(in_train.sum())
test_data = test_data.loc[~in_train]
test_audit["retained_rows"] = len(test_data)
test_data = test_data[["id", "prompt", "label", "adversarial"]].sort_values("id").reset_index(drop=True)

removals = ["excluded_category_rows", "missing_label_rows", "empty_prompt_rows",
            "contradictory_label_rows", "duplicate_rows", "train_overlap_rows", "retained_rows"]
assert sum(test_audit[key] for key in removals) == test_audit["original_rows"]
if not use_toy_data:
    expected_test = {"original_rows": 1725, "excluded_category_rows": 41, "missing_label_rows": 26,
                     "train_overlap_rows": 0, "retained_rows": 1658}
    mismatches = {key: (test_audit[key], value) for key, value in expected_test.items()
                  if test_audit[key] != value}
    assert not mismatches, f"Test audit mismatch (got, expected): {mismatches}"
test_audit["harmful_prevalence"] = float(test_data["label"].mean())
test_audit["adversarial_share"] = float(test_data["adversarial"].mean())
display(pd.Series(test_audit, name="Test audit"))

In [ ]:
model_names = ["logistic", "mlp1", "mlp2"]

def fit_and_predict(model_name, X_train, y_train, X_eval, model_seed):
    """Train one system with the fixed settings and return [unharmful, harmful] probabilities."""
    if model_name == "logistic":
        model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1000, random_state=model_seed)
        model.fit(X_train, y_train)
        return model.predict_proba(X_eval)
    torch.manual_seed(model_seed)
    if model_name == "mlp1":
        model = part1.smlp.SparseMLP(X_train.shape[1], hidden_dim, 2, dropout)
    else:
        model = TwoHiddenMLP(X_train.shape[1], hidden_dim, second_hidden_dim, dropout)
    train_mlp(model, X_train, y_train, epochs=num_epochs, seed=model_seed, batch_size=batch_size,
              learning_rate=learning_rate, weight_decay=weight_decay)
    return predict_mlp(model, X_eval)

test_texts = test_data["prompt"].tolist()
y_test = test_data["label"].to_numpy()
adversarial = test_data["adversarial"].to_numpy(dtype=bool)
test_score_rows, test_prediction_rows = [], []
started_all = time.perf_counter()

for run, sample_seed in enumerate(sample_seeds):
    # Train on the whole balanced sample of this run: no folds, nothing held back.
    run_sample = sample_balanced(data, sample_seed)
    train_texts = run_sample["prompt"].tolist()
    y_train = run_sample["label"].to_numpy()
    tokenizer = part1.experiment.train_tokenizer(train_texts, vocab_size, min_frequency)
    encoders = {"indicators": TokenIndicators(tokenizer), "bow": BagOfWords(tokenizer),
                "tfidf": TFIDF(tokenizer).fit(train_texts)}

    for representation, encoder in encoders.items():
        X_train = encode_texts(train_texts, encoder)
        X_test = encode_texts(test_texts, encoder)
        for model_name in model_names:
            probabilities = fit_and_predict(model_name, X_train, y_train, X_test, sample_seed)
            assert np.isfinite(probabilities).all()
            np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)
            predictions = probabilities.argmax(axis=1)
            row = {"status": run_label, "run": run, "sample_seed": sample_seed, "model": model_name,
                   "representation": representation, **classification_scores(y_test, predictions)}
            for subset, mask in [("adversarial", adversarial), ("vanilla", ~adversarial)]:
                row[f"macro_f1_{subset}"] = classification_scores(y_test[mask], predictions[mask])["macro_f1"]
            test_score_rows.append(row)
            test_prediction_rows.append(pd.DataFrame({
                "run": run, "id": test_data["id"], "model": model_name, "representation": representation,
                "gold": y_test, "prediction": predictions, "probability_harmful": probabilities[:, 1],
                "adversarial": adversarial, "status": run_label}))
    print(f"Run {run + 1}/{num_runs} done ({time.perf_counter() - started_all:.0f}s elapsed)")

test_scores = pd.DataFrame(test_score_rows)
test_predictions = pd.concat(test_prediction_rows, ignore_index=True)
assert len(test_scores) == num_runs * 9
assert len(test_predictions) == num_runs * 9 * len(test_data)
print(run_label, f"— completed all {len(test_scores)} test fits")

In [ ]:
test_scores["system"] = test_scores["model"] + "/" + test_scores["representation"]
test_summary = test_scores.groupby("system").agg(
    test_macro_f1=("macro_f1", "mean"), test_macro_f1_std=("macro_f1", "std"),
    test_accuracy=("accuracy", "mean"), test_harmful_recall=("harmful_recall", "mean"),
    test_benign_fpr=("benign_fpr", "mean"),
    test_f1_adversarial=("macro_f1_adversarial", "mean"), test_f1_vanilla=("macro_f1_vanilla", "mean"),
)
# Put the CV estimate next to the test score, if the CV sections ran in this session.
if "summary" in globals():
    cv = summary.assign(system=summary["model"] + "/" + summary["representation"]).set_index("system")
    test_summary.insert(0, "cv_macro_f1", cv["macro_f1_mean"])
    test_summary["cv_minus_test"] = test_summary["cv_macro_f1"] - test_summary["test_macro_f1"]
test_summary = test_summary.sort_values("test_macro_f1", ascending=False)
display(test_summary.round(4))

# Rank of each system within each training run (1 = best test macro-F1).
test_ranks = test_scores.pivot_table(index="system", columns="run", values="macro_f1").rank(ascending=False).astype(int)
test_ranks["mean_rank"] = test_ranks.mean(axis=1)
display(test_ranks.sort_values("mean_rank"))

In [ ]:
def macro_f1_rows(gold, predictions):
    """Macro-F1 for each row of 2-D gold/prediction arrays (one row per bootstrap sample)."""
    tp = ((gold == 1) & (predictions == 1)).sum(axis=1)
    tn = ((gold == 0) & (predictions == 0)).sum(axis=1)
    fp = ((gold == 0) & (predictions == 1)).sum(axis=1)
    fn = ((gold == 1) & (predictions == 0)).sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        f1_harmful = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
        f1_unharmful = np.where(2 * tn + fp + fn > 0, 2 * tn / (2 * tn + fp + fn), 0.0)
    return (f1_harmful + f1_unharmful) / 2

def system_test_predictions(model, representation):
    """Test predictions as a (runs, prompts) array, in test_data order."""
    subset = test_predictions[(test_predictions.model == model) &
                              (test_predictions.representation == representation)]
    return subset.pivot(index="run", columns="id", values="prediction")[test_data["id"]].to_numpy()

# The same two contrasts as section 11, chosen before seeing the test set.
test_comparisons = [
    (("logistic", "indicators"), ("logistic", "tfidf")),
    (("logistic", "tfidf"), ("mlp1", "tfidf")),
]
num_bootstrap = 2000
bootstrap_rng = np.random.default_rng(seed)
boot_indices = bootstrap_rng.integers(0, len(test_data), size=(num_bootstrap, len(test_data)))
gold_boot = y_test[boot_indices]

test_comparison_rows = []
for system_a, system_b in test_comparisons:
    pred_a, pred_b = system_test_predictions(*system_a), system_test_predictions(*system_b)
    # Per bootstrap sample of test prompts: B - A macro-F1, averaged over the training runs.
    differences = np.mean([macro_f1_rows(gold_boot, pred_b[r][boot_indices]) -
                           macro_f1_rows(gold_boot, pred_a[r][boot_indices]) for r in range(num_runs)], axis=0)
    observed = float(np.mean([classification_scores(y_test, pred_b[r])["macro_f1"] -
                              classification_scores(y_test, pred_a[r])["macro_f1"] for r in range(num_runs)]))
    test_comparison_rows.append({
        "A": "/".join(system_a), "B": "/".join(system_b), "mean_B_minus_A": observed, "rope": rope,
        "ci95_low": float(np.quantile(differences, 0.025)), "ci95_high": float(np.quantile(differences, 0.975)),
        "P_A_wins": float((differences < -rope).mean()),
        "P_equivalent": float((np.abs(differences) <= rope).mean()),
        "P_B_wins": float((differences > rope).mean()),
    })

test_rope = pd.DataFrame(test_comparison_rows)
display(test_rope.round(4))

In [ ]:
test_scores.to_csv(results_folder / "p1_test_scores.csv", index=False)
test_summary.to_csv(results_folder / "p1_test_summary.csv")
test_ranks.to_csv(results_folder / "p1_test_ranks.csv")
test_predictions.to_csv(results_folder / "p1_test_predictions.csv", index=False)
test_rope.to_csv(results_folder / "p1_test_rope.csv", index=False)
(results_folder / "test_audit.json").write_text(json.dumps(test_audit, indent=2))

# Record that the official test has now been used.
settings_path = results_folder / "settings.json"
test_settings = (json.loads(settings_path.read_text()) if settings_path.exists() else
                 {"run_id": run_id, "status": run_label, "sample_seeds": sample_seeds,
                  "examples_per_class": examples_per_class})
test_settings.update({"official_test_accessed": True, "test_split": "wildguardtest",
                      "test_bootstrap_samples": num_bootstrap,
                      "test_evaluated_at": datetime.now(timezone.utc).isoformat()})
settings_path.write_text(json.dumps(test_settings, indent=2))
print("Saved test results:", results_folder.relative_to(root))

## Notes

Sections 9–12 describe five-fold CV repeated on five balanced samples of filtered training data. Section 14 is the single evaluation on the official WildGuardMix test set with fixed settings. Related scenarios, label ambiguity and repeated model comparisons limit what we can conclude.

**Reuse:** the tokenizer, sparse MLP, batching/training utilities and Bayesian analysis come from the course repository. Project additions are counts, TF-IDF, the second hidden layer and the experiment code. The indicator helper also fixes the course encoder's vocabulary-length lookup.